# Auditable Wikipedia RAG-ETL Pipeline with FAISS IVF
## Presentation Notebook

> **This notebook is a presentation controller.**
> It calls existing `src/` scripts via subprocess and reads monitoring evidence from DuckDB.
> It does not re-implement the pipeline.
>
> **Estimated presentation time: 17–18 minutes**
> Run cells top-to-bottom. Pipeline commands (Sections 8, 11, 12) print live output.

In [ ]:
import os, sys, json, shlex, subprocess
from pathlib import Path
import duckdb
import pandas as pd

# Navigate to project root (notebook lives in notebooks/)
os.chdir(Path("..").resolve())
PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

PYTHON       = sys.executable
LIVE_DB      = PROJECT_ROOT / "outputs/live_demo/wiki_demo.duckdb"
FULL_DB      = PROJECT_ROOT / "outputs/full/wiki.duckdb"
LIVE_INITIAL = PROJECT_ROOT / "data/live_demo/initial_sample.jsonl"
LIVE_UPDATE  = PROJECT_ROOT / "data/live_demo/update_sample.jsonl"
LIVE_INDEX   = PROJECT_ROOT / "outputs/live_demo/faiss/index.faiss"
FULL_INDEX   = PROJECT_ROOT / "outputs/full/faiss/index.faiss"
SRC_DIR      = PROJECT_ROOT / "src"

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# ── run_cmd: run a shell command from project root ──────────
def run_cmd(cmd):
    parts = shlex.split(cmd)
    if parts[0] == "python":
        parts[0] = PYTHON
    print(f"$ {' '.join(str(p) for p in parts)}")
    print("─" * 70)
    subprocess.run(parts, cwd=str(PROJECT_ROOT))
    print("─" * 70)

# ── connect_db: open DuckDB read-only (None if missing) ─────
def connect_db(db_path):
    if not Path(db_path).exists():
        print(f"Database not found: {db_path}")
        return None
    return duckdb.connect(str(db_path), read_only=True)

# ── query_df: SQL → DataFrame with error guard ──────────────
def query_df(con, sql):
    try:
        return con.execute(sql).df()
    except Exception as e:
        print(f"Query error: {e}")
        return pd.DataFrame()

# ── show_table: display a table if it exists ────────────────
def show_table(con, table_name, limit=10):
    if con is None:
        print("No database connection.")
        return
    try:
        tables = [r[0] for r in con.execute("SHOW TABLES").fetchall()]
        if table_name not in tables:
            print(f"Table not found: {table_name}")
            return
        df = con.execute(f"SELECT * FROM {table_name} LIMIT {limit}").df()
        display(df)
    except Exception as e:
        print(f"Error showing '{table_name}': {e}")

# ── code_excerpt: print lines around a search string ────────
def code_excerpt(path, search, before=10, after=25):
    p = PROJECT_ROOT / path
    if not p.exists():
        print(f"File not found: {path}")
        return
    lines = p.read_text().splitlines()
    idx = next((i for i, ln in enumerate(lines) if search in ln), None)
    if idx is None:
        print(f"Search term not found: '{search}' in {p.name}")
        return
    s = max(0, idx - before)
    e = min(len(lines), idx + after)
    print(f"# {p.name}  (lines {s+1}–{e})")
    print("─" * 60)
    for i in range(s, e):
        marker = "►" if i == idx else " "
        print(f"{marker} {i+1:4d}│ {lines[i]}")

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {PYTHON}")


---
# 1 · Project Framing

> **"My project is an auditable ETL pipeline that incrementally maintains a Wikipedia-based
> FAISS IVF retrieval system. The goal is not only to retrieve relevant Wikipedia chunks,
> but to show how the data system behaves when documents are added, updated, duplicated,
> or corrupted."**

> **This is an ETL project first, and a RAG system second.**

---

### End-to-end data flow

```
Raw Wikipedia JSON
    → [INGEST]   raw_documents          (staging — verbatim copy)
    → [CLEAN]    rejected_documents     (staging — bad rows quarantined)
    → [CURATE]   clean_documents        (curation — SCD Type 2 versioning)
    → [CHUNK]    fact_chunks            (curation — 384-token sliding window)
    → [EMBED]    fact_embeddings        (curation — E5-base-v2, normalised)
    → [INDEX]    FAISS IVF index        (serving — derived from active embeddings)
    → [QUERY]    Retrieval results      (serving — resolved back through DuckDB)
    → [MONITOR]  runs / row_count_reconciliation / audit_results / latency_logs
```

Two boundaries matter:
- **Staging** (`raw_documents`, `rejected_documents`) — append-only, never modified after insert.
- **Curation** (`clean_documents`, `fact_chunks`, `fact_embeddings`) — SCD Type 2,
  exactly one active version per document at any time.

---
# 2 · Rubric Requirements & Execution Tracks

| Requirement | Implementation |
|-------------|---------------|
| One-command full rebuild | `src/build.py --mode full` |
| Augmented / incremental build | `src/build.py --mode augmented` |
| Idempotency with audit artifacts | `src/audit.py` → `audit_results` table |
| Monitoring evidence | `runs`, `row_count_reconciliation`, `latency_logs` |
| Failure catalog | `rejected_documents` with `reason_code` |
| FAISS IVF retrieval (AI bonus) | `src/index_ivf.py` + `src/retrieve.py` |

### Two execution tracks

| Track | Dataset | Speed | Purpose |
|-------|---------|-------|---------|
| **Full offline** | `data/full/raw_wiki.jsonl` | Slow (run before presentation) | Demonstrates scale |
| **Live demo** | `data/live_demo/initial_sample.jsonl` (40 docs) | Fast (runs live) | Demonstrates all ETL behaviors |

Both tracks use **identical source code**, **identical schema**, and **identical pipeline logic**.

In [ ]:
# Verify that live demo files exist
for label, path in [
    ("LIVE_INITIAL", LIVE_INITIAL),
    ("LIVE_UPDATE",  LIVE_UPDATE),
    ("LIVE_DB",      LIVE_DB),
    ("LIVE_INDEX",   LIVE_INDEX),
    ("FULL_DB",      FULL_DB),
    ("FULL_INDEX",   FULL_INDEX),
]:
    status = "✓" if Path(path).exists() else "✗ MISSING"
    print(f"  {status}  {label:14s}  {path}")


In [ ]:
# build.py orchestrates both modes (search: "run_pipeline")
code_excerpt("src/build.py", "run_pipeline", before=0, after=28)


In [ ]:
# make_live_demo_data.py creates the 4-doc update batch (search: "revised_doc")
code_excerpt("src/make_live_demo_data.py", "revised_doc", before=10, after=25)


---
# 3 · Architecture Overview

> **"DuckDB is the system of record for document lifecycle, chunks, metadata,
> monitoring, and audit artifacts.
> FAISS is a derived serving index built from the active chunk embeddings."**

### Source modules

| Module | Responsibility |
|--------|---------------|
| `src/ingest.py` | JSONL → `raw_documents` |
| `src/clean.py` | Validate, clean text, quarantine bad rows → `rejected_documents` |
| `src/curate.py` | SCD Type 2: NEW / UPDATED / UNCHANGED → `clean_documents` |
| `src/chunk.py` | Sliding-window tokenisation → `fact_chunks` |
| `src/embed.py` | E5-base-v2 encoding → `fact_embeddings` + `.npy` files |
| `src/index_ivf.py` | Train + build IVFFlat → `faiss_index_registry` + `index.faiss` |
| `src/retrieve.py` | Embed query → FAISS search → resolve chunks via DuckDB |
| `src/audit.py` | Cross-run idempotency checks → `audit_results` |
| `src/monitoring.py` | Schema init, run lifecycle, reconciliation, latency logs |

In [ ]:
# List all source modules
print("src/ modules:")
for f in sorted(SRC_DIR.glob("*.py")):
    print(f"  {f.name}")


In [ ]:
# Orchestration entry point — [STAGE] labels show pipeline order
code_excerpt("src/build.py", "[STAGE] INGEST", before=2, after=40)


---
# 4 · Data Grain and DuckDB Schema

| Table | Grain |
|-------|-------|
| `raw_documents` | one row = one raw Wikipedia document per run |
| `clean_documents` | one row = one cleaned document **version** (SCD Type 2) |
| `fact_chunks` | one row = one E5-tokenised chunk from one active document version |
| `fact_embeddings` | one row = one chunk embedding (vectors stored in `.npy` files) |

> **The main retrieval fact table is `fact_chunks`.
> Core grain: one chunk from one active document version.**

Supporting tables: `runs`, `rejected_documents`, `row_count_reconciliation`,
`audit_results`, `faiss_index_registry`, `retrieval_eval`, `latency_logs`.

In [ ]:
con = connect_db(LIVE_DB)
if con:
    print("DuckDB tables in wiki_demo.duckdb:")
    for row in con.execute("SHOW TABLES").fetchall():
        cnt = con.execute(f"SELECT COUNT(*) FROM {row[0]}").fetchone()[0]
        print(f"  {row[0]:<35s}  {cnt:>6d} rows")
    con.close()


In [ ]:
# Schema created in src/monitoring.py (search: "SCHEMA_SQL")
code_excerpt("src/monitoring.py", "SCHEMA_SQL", before=0, after=40)


In [ ]:
con = connect_db(LIVE_DB)
if con:
    print("── raw_documents columns ──")
    display(con.execute("DESCRIBE raw_documents").df())
    print("\n── clean_documents columns ──")
    display(con.execute("DESCRIBE clean_documents").df())
    print("\n── fact_chunks columns ──")
    display(con.execute("DESCRIBE fact_chunks").df())
    con.close()


---
# 5 · Cleaning and Update Contract

> **"Wikipedia has no revision ID field. The system uses a deterministic SHA-256
> content hash of the cleaned text as the version detector."**

### Update rules

| Incoming doc | Action |
|-------------|--------|
| New `id` | NEW — insert active row, generate chunks and embeddings |
| Same `id` + same `content_hash` | UNCHANGED — nothing written |
| Same `id` + different `content_hash` | UPDATED — old row deactivated, new row + chunks + embeddings |
| Empty / null text | REJECTED — written to `rejected_documents`, excluded from indexing |

> **Exactly one active version per `doc_id` at all times.
> Duplicate submissions do not create new chunks or embeddings.**

In [ ]:
# Preview initial sample and update batch
print(f"initial_sample.jsonl  ({LIVE_INITIAL}):")
if LIVE_INITIAL.exists():
    docs = [json.loads(l) for l in LIVE_INITIAL.read_text().splitlines() if l.strip()]
    df_init = pd.DataFrame([{"id": d["id"], "title": d["title"],
                              "text_len": len(d.get("text",""))} for d in docs])
    display(df_init.head(5))
    print(f"  total: {len(docs)} documents")

print(f"\nupdate_sample.jsonl  ({LIVE_UPDATE}):")
if LIVE_UPDATE.exists():
    udocs = [json.loads(l) for l in LIVE_UPDATE.read_text().splitlines() if l.strip()]
    df_upd = pd.DataFrame([{"id": d["id"], "title": d.get("title",""),
                             "text_len": len(d.get("text") or ""),
                             "has_text": bool((d.get("text") or "").strip())}
                           for d in udocs])
    display(df_upd)
    print("  (corrupted doc = has_text: False)")


In [ ]:
con = connect_db(LIVE_DB)
if con:
    print("── clean_documents (active versions) ──")
    display(query_df(con, '''
        SELECT doc_id, title,
               content_hash[:12] || '...' AS hash_prefix,
               is_active, valid_from_run_id, valid_to_run_id
        FROM clean_documents
        ORDER BY doc_id
        LIMIT 6
    '''))

    print("\n── rejected_documents ──")
    show_table(con, "rejected_documents")
    con.close()


In [ ]:
# validate_and_clean_raw: validates & computes content_hash
code_excerpt("src/clean.py", "validate_and_clean_raw", before=0, after=30)


In [ ]:
# SCD logic in curate_documents: deactivation on update
code_excerpt("src/curate.py", "is_active=FALSE", before=12, after=20)


---
# 6 · Chunking and FAISS IVF Design

### Chunking policy

| Parameter | Value |
|-----------|-------|
| Input text | `"{title}: {cleaned_text}"` |
| Tokenizer | `intfloat/e5-base-v2` |
| Chunk size | 384 tokens |
| Overlap | 64 tokens |
| Stride | 320 tokens |
| Min chunk length | 32 tokens |
| Chunk ID | `sha256(doc_id | content_hash | start_tok | end_tok)` — deterministic |

Each chunk stores `doc_id`, `content_hash`, `start_tok`, `end_tok` — full provenance back to the source document version.

### FAISS IVF

> **"The FAISS IVFFlat index is a derived serving artifact built from active chunk
> embeddings. Full rebuild retrains and rebuilds it from scratch. Augmented build
> re-embeds only new or updated documents, then rebuilds the IVF from the full active
> embedding set to avoid stale vectors and index drift. The ETL delta is confirmed
> by `new_doc_count` and `updated_doc_count` in the `runs` table."**

In [ ]:
con = connect_db(LIVE_DB)
if con:
    n = con.execute("SELECT COUNT(*) FROM fact_chunks WHERE is_active=TRUE").fetchone()[0]
    print(f"Active fact_chunks: {n}")
    print()
    display(query_df(con, '''
        SELECT chunk_id[:12]||'...' AS chunk_id,
               doc_id, chunk_index, start_tok, end_tok,
               chunk_token_len,
               LEFT(chunk_text, 60) || '...' AS preview,
               is_active
        FROM fact_chunks
        WHERE is_active=TRUE
        ORDER BY doc_id, chunk_index
        LIMIT 5
    '''))

    print("\n── faiss_index_registry ──")
    show_table(con, "faiss_index_registry")
    con.close()


In [ ]:
# Sliding window in chunk_document (search: "MIN_CHUNK_LEN")
code_excerpt("src/chunk.py", "MIN_CHUNK_LEN", before=10, after=20)


In [ ]:
# E5 passage prefix and normalisation in embed_new_chunks
code_excerpt("src/embed.py", "PASSAGE_PREFIX", before=5, after=15)


In [ ]:
# IVFFlat construction in build_index
code_excerpt("src/index_ivf.py", "IndexIVFFlat", before=8, after=18)


---
# 7 · Live Demo Setup

> **"The live demo uses a 40-document Wikipedia subset so all pipeline stages
> run within the presentation window. The code, schema, chunking logic,
> embedding model, and FAISS IVF configuration are identical to the full corpus run."**

### Update batch composition

| Doc | Scenario |
|-----|----------|
| 1 new document | Unseen `id` → NEW |
| 1 revised document | Same `id`, text with appended sentence → UPDATED |
| 1 duplicate document | Exact copy of an existing doc → UNCHANGED |
| 1 corrupted document | Empty `text` field → REJECTED |

In [ ]:
if LIVE_INITIAL.exists() and LIVE_UPDATE.exists():
    n_init = sum(1 for l in LIVE_INITIAL.read_text().splitlines() if l.strip())
    n_upd  = sum(1 for l in LIVE_UPDATE.read_text().splitlines() if l.strip())
    print(f"initial_sample.jsonl : {n_init} documents")
    print(f"update_sample.jsonl  : {n_upd}  documents")

    print("\nUpdate batch details:")
    upd_docs = [json.loads(l) for l in LIVE_UPDATE.read_text().splitlines() if l.strip()]
    init_ids = {json.loads(l)["id"] for l in LIVE_INITIAL.read_text().splitlines() if l.strip()}

    for d in upd_docs:
        doc_id   = d.get("id", "")
        title    = d.get("title", "")
        text     = d.get("text") or ""
        is_new   = doc_id not in init_ids
        has_text = bool(text.strip())
        print(f"  id={doc_id:<20s}  title={title[:40]:<40s}  "
              f"new={str(is_new):<5}  has_text={has_text}")


---
# 8 · Live Full Rebuild  *(~3–4 min)*

> **"This command rebuilds all layers from raw data:
> raw ingestion → cleaning → SCD document versioning → chunking →
> embedding → FAISS IVF training and indexing → monitoring tables."**

The `[STAGE]` labels in the output correspond directly to the pipeline stages shown in Section 3.

In [ ]:
run_cmd(
    "python src/build.py "
    "--mode full "
    "--dataset-name live_demo "
    "--input data/live_demo/initial_sample.jsonl "
    "--output outputs/live_demo "
    "--db outputs/live_demo/wiki_demo.duckdb "
    "--index-type ivf"
)


---
# 9 · Monitoring After Full Rebuild

> **"Every build is logged in the `runs` table.
> `row_count_reconciliation` checks that active chunks == active embeddings == index vectors.
> `latency_logs` captures per-stage timing."**

In [ ]:
con = connect_db(LIVE_DB)
if con:
    print("── runs ──")
    RUN_COLS = [
        "run_id", "mode", "status",
        "raw_doc_count", "stg_valid_count", "rejected_doc_count",
        "active_doc_count", "active_chunk_count",
        "active_embedding_count", "index_vector_count"
    ]
    try:
        display(con.execute(f"SELECT {', '.join(RUN_COLS)} FROM runs ORDER BY run_id").df())
    except Exception:
        display(con.execute("SELECT * FROM runs ORDER BY run_id LIMIT 5").df())

    print("\n── row_count_reconciliation ──")
    show_table(con, "row_count_reconciliation")

    print("\n── latency_logs (latest run) ──")
    try:
        latest_run = con.execute("SELECT MAX(run_id) FROM runs WHERE status='success'").fetchone()[0]
        display(con.execute(f'''
            SELECT stage_name, duration_seconds, row_count
            FROM latency_logs WHERE run_id={latest_run}
            ORDER BY start_time
        ''').df())
    except Exception as e:
        print(f"latency_logs unavailable: {e}")
    con.close()


In [ ]:
# Monitoring functions in src/monitoring.py
code_excerpt("src/monitoring.py", "write_reconciliation", before=0, after=30)


---
# 10 · Retrieval Before Update

> **"Retrieval uses E5-base-v2 with a `query:` prefix to embed the question.
> FAISS returns vector IDs, which are resolved through `vector_map.npy` to `chunk_id`s,
> then joined in DuckDB to return title, URL, and chunk text."**

In [ ]:
run_cmd(
    'python src/retrieve.py '
    '--db outputs/live_demo/wiki_demo.duckdb '
    '--index outputs/live_demo/faiss/index.faiss '
    '--query "What is artificial intelligence?" '
    '--top-k 5'
)


In [ ]:
# Query embedding + FAISS search + vector_map resolution
code_excerpt("src/retrieve.py", "vector_map", before=10, after=20)


---
# 11 · Idempotency Audit  *(~3–4 min)*

> **"Idempotency is not just claimed — it is verified by comparing two consecutive
> full rebuilds over the same input and writing the results to `audit_results`."**

### Five audit checks

| Check | What it compares |
|-------|-----------------|
| `active_doc_count` | Total active documents in runs table |
| `active_chunk_count` | Total active chunks in runs table |
| `active_embedding_count` | Total active embeddings in runs table |
| `index_vector_count` | FAISS index size in runs table |
| `chunk_hash_checksum` | SHA-256 of sorted concatenation of all active `chunk_hash` values |

Running the same full build twice must produce `PASS` on all 5 checks.

In [ ]:
# Second full rebuild — same command, same input
run_cmd(
    "python src/build.py "
    "--mode full "
    "--dataset-name live_demo "
    "--input data/live_demo/initial_sample.jsonl "
    "--output outputs/live_demo "
    "--db outputs/live_demo/wiki_demo.duckdb "
    "--index-type ivf"
)


In [ ]:
con = connect_db(LIVE_DB)
if con:
    print("── audit_results ──")
    df_audit = query_df(con, '''
        SELECT run_id_a, run_id_b, audit_type, result, details
        FROM audit_results
        ORDER BY created_at DESC
        LIMIT 10
    ''')
    display(df_audit)

    if not df_audit.empty:
        passes = (df_audit["result"] == "PASS").sum()
        fails  = (df_audit["result"] == "FAIL").sum()
        print(f"\n  ✓ {passes} PASS   ✗ {fails} FAIL")

    print("\n── runs after second rebuild ──")
    try:
        display(con.execute('''
            SELECT run_id, mode, status,
                   new_doc_count, unchanged_doc_count,
                   new_chunk_count, active_chunk_count,
                   index_vector_count
            FROM runs ORDER BY run_id DESC LIMIT 4
        ''').df())
    except Exception:
        show_table(con, "runs", limit=4)
    con.close()


In [ ]:
# Audit implementation: chunk_hash_checksum in src/audit.py
code_excerpt("src/audit.py", "chunk_hash_checksum", before=8, after=15)


---
# 12 · Augmented Build  *(~1–2 min)*

> **"The augmented build processes only the 4-document update batch.
> It detects each document's status through the SCD logic and only
> creates new chunks and embeddings for new or updated documents.
> The FAISS index is refreshed from the full active embedding set."**

Expected outcomes from `update_sample.jsonl`:
- 1 NEW doc → chunked and embedded
- 1 UPDATED doc → old chunks/embeddings deactivated, new ones created
- 1 UNCHANGED doc → nothing written
- 1 CORRUPTED doc → written to `rejected_documents`

In [ ]:
run_cmd(
    "python src/build.py "
    "--mode augmented "
    "--dataset-name live_demo "
    "--input data/live_demo/update_sample.jsonl "
    "--output outputs/live_demo "
    "--db outputs/live_demo/wiki_demo.duckdb "
    "--index-type ivf"
)


---
# 13 · Delta After Augmented Build

> **"The `runs` table shows exactly what was recomputed and what was preserved.
> `row_count_reconciliation` confirms the index is consistent with the active embedding set."**

In [ ]:
con = connect_db(LIVE_DB)
if con:
    print("── runs (all builds) ──")
    DELTA_COLS = [
        "run_id", "mode",
        "raw_doc_count", "new_doc_count", "updated_doc_count",
        "duplicate_doc_count", "unchanged_doc_count", "rejected_doc_count",
        "new_chunk_count", "new_embedding_count",
        "active_chunk_count", "index_vector_count"
    ]
    try:
        display(con.execute(f"SELECT {', '.join(DELTA_COLS)} FROM runs ORDER BY run_id").df())
    except Exception:
        display(con.execute("SELECT * FROM runs ORDER BY run_id LIMIT 6").df())

    print("\n── row_count_reconciliation ──")
    show_table(con, "row_count_reconciliation")

    print("\n── rejected_documents ──")
    show_table(con, "rejected_documents")
    con.close()


In [ ]:
con = connect_db(LIVE_DB)
if con:
    # Show the SCD version history for the revised document
    revised_id = None
    if LIVE_INITIAL.exists() and LIVE_UPDATE.exists():
        init_docs = {json.loads(l)["id"] for l in LIVE_INITIAL.read_text().splitlines() if l.strip()}
        for l in LIVE_UPDATE.read_text().splitlines():
            if not l.strip(): continue
            d = json.loads(l)
            if d.get("id") in init_docs and (d.get("text") or "").strip():
                revised_id = d["id"]
                break

    if revised_id:
        print(f"── SCD version history for revised doc_id={revised_id} ──")
        display(query_df(con, f'''
            SELECT doc_id, title,
                   content_hash[:12] || '...' AS hash_prefix,
                   source_run_id, is_active,
                   valid_from_run_id, valid_to_run_id
            FROM clean_documents
            WHERE doc_id = '{revised_id}'
            ORDER BY valid_from_run_id
        '''))
        print()
        display(query_df(con, f'''
            SELECT chunk_id[:12]||'...' AS chunk_id,
                   chunk_index, source_run_id,
                   is_active, valid_from_run_id, valid_to_run_id
            FROM fact_chunks
            WHERE doc_id = '{revised_id}'
            ORDER BY source_run_id, chunk_index
        '''))
    con.close()


---
# 14 · Failure Catalog

| Failure | Origin | Detected by | Curated output |
|---------|--------|-------------|----------------|
| **Duplicate** | Re-ingestion of same document | `curate.py` — same `id` + same `content_hash` | Nothing written; not indexed |
| **Revised** | Changed text under same `id` | `curate.py` — `content_hash` mismatch | Old version deactivated; new chunks + embeddings created |
| **Corrupted** | Empty or null `text` field | `clean.py` — `EMPTY_TEXT` validation | Written to `rejected_documents`; excluded from all downstream |

> **No failure mode silently contaminates the retrieval index.
> Each one is either deduplicated, versioned, or quarantined.**

In [ ]:
con = connect_db(LIVE_DB)
if con:
    print("── rejected_documents (full detail) ──")
    show_table(con, "rejected_documents", limit=20)

    print("\n── clean_documents: active vs. deactivated ──")
    display(query_df(con, '''
        SELECT is_active,
               COUNT(*)           AS doc_versions,
               COUNT(DISTINCT doc_id) AS unique_doc_ids
        FROM clean_documents
        GROUP BY is_active
        ORDER BY is_active DESC
    '''))

    print("\n── fact_chunks: active vs. deactivated ──")
    display(query_df(con, '''
        SELECT is_active, COUNT(*) AS chunk_count
        FROM fact_chunks
        GROUP BY is_active
        ORDER BY is_active DESC
    '''))
    con.close()


In [ ]:
# EMPTY_TEXT validation in src/clean.py
code_excerpt("src/clean.py", "EMPTY_TEXT", before=8, after=15)


---
# 15 · Retrieval After Update

> **"After the augmented build, the revised document's new content is embedded
> and indexed. Queries that match the new content now surface the updated chunk."**

The revised doc had this sentence appended to its text:
*"This updated version includes new information about vector databases and retrieval systems."*

In [ ]:
run_cmd(
    'python src/retrieve.py '
    '--db outputs/live_demo/wiki_demo.duckdb '
    '--index outputs/live_demo/faiss/index.faiss '
    '--query "What document discusses vector databases and retrieval systems?" '
    '--top-k 5'
)


In [ ]:
# Also check retrieval_eval table if populated
con = connect_db(LIVE_DB)
if con:
    tables = [r[0] for r in con.execute("SHOW TABLES").fetchall()]
    if "retrieval_eval" in tables:
        n = con.execute("SELECT COUNT(*) FROM retrieval_eval").fetchone()[0]
        if n > 0:
            print("── retrieval_eval ──")
            show_table(con, "retrieval_eval")
        else:
            print("retrieval_eval: table exists but is empty (populated by optional eval script)")
    con.close()


---
# 16 · Full Offline Corpus Evidence

> **"The live demo uses a 40-doc subset for presentation speed.
> I ran the same pipeline on the larger Wikipedia corpus before this presentation.
> The same `src/build.py` command, same schema, same logic — just a larger input file."**

```bash
# Full corpus build (run before presentation)
python src/build.py \
  --mode full \
  --dataset-name full \
  --input data/full/raw_wiki.jsonl \
  --output outputs/full \
  --db outputs/full/wiki.duckdb \
  --index-type ivf
```

In [ ]:
con = connect_db(FULL_DB)
if con:
    print("── full corpus: runs ──")
    try:
        display(con.execute('''
            SELECT run_id, mode, status,
                   raw_doc_count, active_doc_count,
                   active_chunk_count, index_vector_count
            FROM runs ORDER BY run_id
        ''').df())
    except Exception:
        show_table(con, "runs", limit=5)

    print("\n── full corpus: row_count_reconciliation ──")
    show_table(con, "row_count_reconciliation")

    print("\n── full corpus: faiss_index_registry ──")
    show_table(con, "faiss_index_registry")
    con.close()
else:
    print("Full corpus DB not found — run 'src/build.py --mode full --dataset-name full ...' first.")
    print("Live demo results above are sufficient for the demo.")


---
# 17 · Closing Summary

> **"The main contribution of this project is an auditable ETL system for maintaining
> a vector retrieval index under document updates. It supports full rebuilds,
> augmented builds, deterministic chunking, failure handling, monitoring,
> and idempotency audits. FAISS IVF is integrated as a derived serving artifact
> from the curated chunk embedding layer."**

### System properties demonstrated

| Property | Evidence |
|----------|----------|
| Full rebuild | `runs.mode = 'full'`, all stages executed, `runs.status = 'success'` |
| Augmented build | `new_doc_count`, `updated_doc_count` in `runs`; delta confirmed |
| Idempotency | `audit_results` — 5/5 PASS on consecutive full rebuilds |
| Failure handling | `rejected_documents` — corrupted doc quarantined |
| SCD versioning | `clean_documents` — two versions of revised doc, one active |
| Reconciliation | `active_chunks = active_embeddings = index_vectors` in `row_count_reconciliation` |
| Retrieval change | Query for "vector databases" surfaces revised doc post-update |

---

## Q & A

**Q: Why chunk grain?**
> Because retrieval is performed at the chunk level. Documents are stored at document grain for lifecycle management, but embeddings and FAISS vectors are generated per chunk. One document produces many chunks, each independently retrievable.

**Q: Why rebuild IVF during augmented build?**
> The ETL computation is augmented — only new or updated documents are cleaned, chunked, and embedded. The IVF index is a derived serving artifact. It is rebuilt from the full active embedding set to avoid stale vectors and index drift. `new_doc_count` and `updated_doc_count` in `runs` confirm that ETL-level delta processing occurred.

**Q: How are updates detected without a revision ID?**
> By deterministic SHA-256 content hash of the cleaned text. Same ID + same hash → unchanged. Same ID + different hash → updated.

**Q: How is double-counting prevented?**
> Exactly one active version per `doc_id` at all times. The SCD Type 2 logic in `curate_documents` deactivates the old row before inserting the new one. Duplicate submissions (same ID + same hash) do not create new chunks or embeddings.

**Q: How is idempotency verified?**
> `src/audit.py` compares two runs on: active document count, active chunk count, sorted chunk-hash checksum, embedding count, and index vector count. Results are persisted in `audit_results`. A verbal claim is insufficient — the artifact must exist.

**Q: What happens to corrupted rows?**
> They fail validation in `validate_and_clean_raw` with `reason_code='EMPTY_TEXT'`, are written to `rejected_documents`, and are excluded from chunking, embedding, FAISS indexing, and retrieval.